# 82514 · Sesión S16 — Formación de imagen y taller de calibración de la cámara del equipo

**Bloque 6** · jueves 12 de noviembre de 2026 · 2 h · IQS Universitat Ramon Llull

Cada uno calibra, solo, la cámara de su equipo, una unidad simulada, y lleva el píxel a la mesa del 2R y a sus ángulos.

| | |
|---|---|
| **En clase** | Secciones 1 a 4: las tres fases del taller, de doce minutos cada una. |
| **En casa** | Sección 5 y ejercicios 1 a 3; las soluciones, al final. |
| **Entrega** | Al acabar la fase 3, antes de la puesta en común: este cuaderno ejecutado (Archivo → Descargar → Descargar .ipynb) y `calibracion_E<n><letra>.json`, a la Tarea «Calibración de la cámara del equipo» de Moodle. Es individual: cada uno sube los suyos. Admite hasta las 23:59 del jueves 12. |

Cada paso tiene tres celdas: **▶ Parámetros**, lo único que se cambia; **⚙ Cálculo**, que no hace falta tocar; y **✔ Resultado**, lo que sale. En las celdas **✎ Completad**, el hueco está marcado con `None`.

---

In [ ]:
# @title ⚙ Instalación · ejecutad esta celda primero
# Ejecutar una sola vez. En Colab tarda unos veinte segundos.
import importlib, subprocess, sys

def asegurar(mods):
    faltan = []
    for pip_name, import_name in mods:
        try:
            importlib.import_module(import_name)
        except ImportError:
            faltan.append(pip_name)
    if faltan:
        print('Instalando:', ' '.join(faltan))
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + faltan, check=False)
    else:
        print('Todo instalado ya.')

asegurar([('numpy', 'numpy'), ('matplotlib', 'matplotlib')])

OPENCV = '4.14.0.94'          # la versión de todo el bloque 6 (cv2.__version__ = '4.14.0')

def opencv_activo():
    """Versión de OpenCV que se importaría, mirada en otro proceso. Colab trae tres paquetes de
    OpenCV que comparten la carpeta cv2, y lo que dice pip no basta para saber cuál se carga."""
    r = subprocess.run([sys.executable, '-c', 'import cv2; print(cv2.__version__)'], capture_output=True, text=True)
    return r.stdout.strip()

def fijar_opencv():
    if opencv_activo() != OPENCV.rsplit('.', 1)[0]:
        print('Instalando OpenCV', OPENCV, '(unos veinte segundos)...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'opencv-python-headless=={OPENCV}'], check=False)

fijar_opencv()
import os, json, time, datetime, tempfile, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import cv2

EN_COLAB = 'google.colab' in sys.modules
np.set_printoptions(precision=4, suppress=True)
IQS_AZUL, IQS_VERDE = '#1B2A80', '#1FA355'
GRIS, AMBAR, ROJO = '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def ejes(*axs):
    for ax in axs:
        ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para los resultados."""
    s = f'{x:.{n}f}'
    if s.startswith('-') and not s.strip('-0.'):      # un negativo que se redondea a cero, sin signo
        s = s[1:]
    return s.replace('.', ',').replace('-', '−')

print('OpenCV', cv2.__version__, '· numpy', np.__version__, '· en Colab:', EN_COLAB)
if cv2.__version__ != OPENCV.rsplit('.', 1)[0]:
    print('AVISO: este cuaderno se probó con OpenCV', OPENCV, '. Entorno de ejecución → Reiniciar sesión, y vuelve a ejecutar esta celda.')
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy 2.4.6; con otra versión las cifras pueden cambiar en la última decimal.')

## 1. La cámara de vuestro equipo y el puesto de picking

*En clase, fase 1, sin deteneros · diapositiva 14*

La cámara, cenital a 1 m sobre la mesa del 2R; cada unidad se aparta de la hoja de datos.
Cada uno trabaja solo, con su portátil. Poned `EQUIPO`, `LETRA` y `NOMBRE` y ejecutad el simulador sin abrirlo: su verdad se desvela en S20.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# vuestro equipo del proyecto, de 1 a 5 (el 0 es la cámara del curso, para casa)
EQUIPO = 1  # @param {type:"number"}
# vuestra letra en el equipo, por orden alfabético del primer apellido: A, B, C, D (E, si sois cinco);
# los del equipo calibráis la misma unidad, cada uno con sus fotos
LETRA = 'A'  # @param ["A", "B", "C", "D", "E"]
# vuestro nombre y apellidos
NOMBRE = 'Nombre Apellido1 Apellido2'  # @param {type:"string"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo · el simulador: ejecutadlo sin abrirlo
# ── El simulador de la cámara de vuestro equipo y del puesto de picking ──────────────────────────────
# Ejecutad esta celda sin deteneros: no hay nada que cambiar. Simula lo que haríais con una cámara de verdad:
# fotografiar el tablero, apoyarlo en la mesa y medir marcas de la mesa. Física de calc_bloque.py (B6, §3).
import json, zlib, base64

W, H = 640, 480                        # la imagen: 640 x 480 píxeles
RHO, FOPT = 5e-6, 4e-3                 # hoja de datos: píxel de 5 µm y óptica de 4 mm, es decir, 800 px nominales
NX, NY, LADO = 9, 6, 0.025             # el tablero: 9 x 6 esquinas interiores y cuadros de 25 mm (10 x 7 cuadros)
OBJP = np.zeros((NX*NY, 3)); OBJP[:, :2] = np.mgrid[0:NX, 0:NY].T.reshape(-1, 2)*LADO
CENTRO_TAB = np.array([(NX - 1)*LADO/2, (NY - 1)*LADO/2, 0.0])
A1, A2 = 0.35, 0.20                    # el 2R de B4, en el plano de la mesa
D2R, R2D = np.pi/180, 180/np.pi


def Rx(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def Ry(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def Rz(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def Tmat(R, t):
    M = np.eye(4); M[:3, :3] = R; M[:3, 3] = t
    return M


def Tinv(M):
    """La inversa de S8: la rotación se traspone y la traslación pasa a ser −Rᵀ·t."""
    R, t = M[:3, :3], M[:3, 3]
    return Tmat(R.T, -R.T @ t)


# El puesto de picking (diapositiva 8): la cámara cenital a 1 m sobre la mesa, mirando hacia abajo
T_0C_NOMINAL = Tmat(Rx(np.pi), [0.30, 0.20, 1.00])     # lo que dice el plano del montaje
P_PIEZA, H_PIEZA = np.array([0.40, 0.20, 0.0]), 0.010     # la pieza: un disco de 40 mm y 10 mm de alto
T_0TAB = Tmat(np.eye(3), [0.25, 0.10, 0.0])              # el tablero apoyado en la mesa, con su origen en la marca (0,25; 0,10)
MARCAS = np.array([[x, y, 0.0] for y in np.arange(0, 0.41, 0.1) for x in np.arange(0, 0.61, 0.1)])  # el tapete: 35 marcas

# Las unidades del modelo (la 0 es la cámara del curso). Cada unidad tiene su focal, su centro, su distorsión y
# su montaje, que no son los nominales. Es la verdad que se desvela el jueves 26, en S20: no la decodifiquéis.
_U = 'eNo9kFkOwyAMRO+Sb8fCC0t6laonqXr3zji0ijCE4MebvI92PJ6rNW0SzurJejb1LlgNlnvcj/7HSw5D81yhjuapKR5drZonT1wsrQV3MLuknCFEJrcCiwWKU8Fcl4RhBwqxKU5A3+150/Cli8tZCDpibergBDmeFSUrytJZHIqYybbYUiFdqMBBpw5EMtAVeA8baHa/IERE3T63iu0ZyXD7kPsIS4rVn+kEzcoUBk1PB5ShqWH+y7KVhgA5S6RSoczX5wuk8U/j'
_UNIDADES = {int(k): v for k, v in json.loads(zlib.decompress(base64.b64decode(_U))).items()}
# La letra de cada alumno en su equipo: todos fotografían la misma unidad, cada uno con sus fotos (su semilla, la
# posición de la letra) y con su vista despegada ((7·equipo + desplazamiento) % 15, distinta para cada letra)
LETRAS = 'ABCDE'
_DESPEGADA = {'A': 3, 'B': 10, 'C': 4, 'D': 5, 'E': 9}


class Camara:
    """Una unidad del modelo: K, distorsión y montaje verdaderos. Hace las fotos; no dice su verdad."""

    def __init__(self, equipo, letra='A', ss=2):
        if not (isinstance(letra, str) and len(letra) == 1 and letra in LETRAS):
            raise ValueError(f"La letra es una de 'A', 'B', 'C', 'D' o 'E', entre comillas; no {letra!r}")
        fx, cx, cy, k1, k2, p1, p2, dx, dy, dz, ax, ay, az = _UNIDADES[equipo]
        self._K = np.array([[fx, 0, cx], [0, fx, cy], [0, 0, 1.0]])
        self._d = np.array([k1, k2, p1, p2, 0.0])
        self._T_0C = T_0C_NOMINAL @ Tmat(Rz(az*D2R) @ Ry(ay*D2R) @ Rx(ax*D2R), np.array([dx, dy, dz])*1e-3)
        self.equipo, self.letra = equipo, letra
        self.rng = np.random.default_rng(1000*equipo + LETRAS.index(letra) + 1)
        us = (np.arange(W*ss) + 0.5)/ss - 0.5
        vs = (np.arange(H*ss) + 0.5)/ss - 0.5
        U, V = np.meshgrid(us, vs)
        pts = np.stack([U.ravel(), V.ravel()], -1).reshape(-1, 1, 2)
        n = cv2.undistortPointsIter(pts, self._K, self._d, None, None,
                                    (cv2.TERM_CRITERIA_COUNT | cv2.TERM_CRITERIA_EPS, 60, 1e-12)).reshape(-1, 2)
        self._n = np.vstack([n.T, np.ones(len(n))])      # el rayo de cada subpíxel
        self.ss = ss
        self._doblada = (7*equipo + _DESPEGADA[letra]) % 15   # una de las quince salió con la hoja despegada

    def pose(self, fila):
        """Una fila de la tabla de vistas -> pose del tablero en la cámara (R, t)."""
        Z, inc_x, inc_y, u, v, giro = fila
        R = cv2.Rodrigues(np.array([inc_x, inc_y, 0.0])*D2R)[0] @ Rz(giro*D2R)
        t = Z*(np.linalg.inv(self._K) @ np.array([u, v, 1.0])) - R @ CENTRO_TAB
        return R, t

    def cabe(self, fila, margen=12):
        R, t = self.pose(fila)
        uv = cv2.projectPoints(OBJP, cv2.Rodrigues(R)[0], t, self._K, self._d)[0].reshape(-1, 2)
        return bool(np.all((uv[:, 0] > margen) & (uv[:, 0] < W - margen) & (uv[:, 1] > margen) & (uv[:, 1] < H - margen)))

    def _plano(self, M):
        q = np.linalg.solve(M, self._n)
        return q[0]/q[2], q[1]/q[2], q[2]                 # coordenadas en el plano y 1/profundidad

    def foto(self, R, t, fondo=105, blanco=215, negro=35, blur=0.7, ruido=2.0, reflejo=None, doblez=None):
        """Imagen en gris del tablero con pose (R, t): cada píxel, su rayo, cortado con el plano del tablero.
        reflejo = (x, y, radio) en metros sobre el tablero; doblez = (y del pliegue, ángulo en grados)."""
        X, Y, iz = self._plano(np.column_stack([R[:, 0], R[:, 1], t]))
        delante = iz > 0
        if doblez is not None:                            # la hoja despegada: a partir de y = yf, levantada
            yf, b = doblez[0], doblez[1]*D2R
            e2 = np.array([0, np.cos(b), np.sin(b)])
            X2, S2, iz2 = self._plano(np.column_stack([R[:, 0], R @ e2, t + R @ np.array([0, yf, 0])]))
            d1 = delante & (Y < yf)
            usa2 = (iz2 > 0) & (S2 >= 0) & (~d1 | (iz2 > iz))
            X, Y, delante = np.where(usa2, X2, X), np.where(usa2, yf + S2, Y), d1 | usa2
        ix, iy = np.floor(X/LADO).astype(int) + 1, np.floor(Y/LADO).astype(int) + 1
        en_cuadros = delante & (ix >= 0) & (ix <= NX) & (iy >= 0) & (iy <= NY)
        en_tablero = delante & (X >= -2*LADO) & (X <= (NX + 1)*LADO) & (Y >= -2*LADO) & (Y <= (NY + 1)*LADO)
        img = np.full(X.shape, float(fondo))
        img[en_tablero] = blanco
        img[en_cuadros & ((ix + iy) % 2 == 0)] = negro
        if reflejo is not None:
            cx_, cy_, rad = reflejo
            img[en_tablero & ((X - cx_)**2 + (Y - cy_)**2 < rad**2)] = 255
        ss = self.ss
        img = img.reshape(H*ss, W*ss).reshape(H, ss, W, ss).mean(axis=(1, 3))
        img = cv2.GaussianBlur(img, (0, 0), blur) + self.rng.normal(0, ruido, (H, W))
        return np.clip(np.round(img), 0, 255).astype(np.uint8)

    def fotos_de(self, vistas):
        """Las fotos de una tabla de vistas. Como en el aula, una de ellas sale con la hoja despegada del cartón."""
        return [self.foto(*self.pose(f), doblez=(0.0875, 4.0) if i == self._doblada else None) for i, f in enumerate(vistas)]

    def reservadas(self):
        """Las dos vistas que no entran en el ajuste: inclinadas, en dos zonas distintas de la imagen."""
        return [self.foto(*self.pose(f)) for f in ((0.45, 25, -20, 250, 200, 12), (0.50, -20, 30, 390, 290, -8))]

    def trampas(self):
        """Los tres fallos de siempre: el tablero rasante (a 82°), con un reflejo de 70 mm y cortado por el borde."""
        return [self.foto(*self.pose((0.50, 82, 0, 320, 240, 10))),
                self.foto(*self.pose((0.45, 20, 0, 320, 240, 5)), reflejo=(0.10, 0.06, 0.035)),
                self.foto(*self.pose((0.40, 10, 0, 600, 240, 0)))]

    def foto_mesa(self, fondo=150):
        """El tablero apoyado en la mesa, en T_0TAB, visto desde el montaje verdadero de la cámara."""
        T = Tinv(self._T_0C) @ T_0TAB
        return self.foto(T[:3, :3], T[:3, 3], fondo=fondo)

    def mide(self, P0, sigma=0.1):
        """El píxel donde se ven unos puntos de la mesa (marcas impresas), con su ruido de medida."""
        T = Tinv(self._T_0C)
        uv = cv2.projectPoints(np.atleast_2d(P0).astype(np.float64), cv2.Rodrigues(T[:3, :3])[0], T[:3, 3], self._K, self._d)[0]
        return uv.reshape(-1, 2) + self.rng.normal(0, sigma, (len(np.atleast_2d(P0)), 2))


CRIT = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_MAX_ITER, 50, 1e-4)


def detecta(img):
    """Las 54 esquinas del tablero, con precisión subpíxel; None si no lo encuentra entero."""
    ok, esq = cv2.findChessboardCorners(img, (NX, NY), cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE)
    return cv2.cornerSubPix(img, esq, (5, 5), (-1, -1), CRIT) if ok else None


def calibra(esquinas):
    """K, la distorsión (k1, k2, p1, p2; k3 = 0) y una pose por vista, por mínimos cuadrados (LM).
    Devuelve el RMS de todas las esquinas, K, la distorsión y el RMS de cada vista, calculado por OpenCV."""
    obj = [OBJP.astype(np.float32)]*len(esquinas)
    sal = cv2.calibrateCameraExtended(obj, esquinas, (W, H), None, None, flags=cv2.CALIB_FIX_K3)
    rms, K, d, por_vista = sal[0], sal[1], sal[2].ravel(), sal[-1].ravel()
    return rms, K, d, por_vista


def err_reservada(K, d, esq):
    """RMS de reproyección de una vista que no entró en el ajuste: su pose sale de solvePnP con K y d fijas."""
    ok, rv, tv = cv2.solvePnP(OBJP, esq, K, d)
    uv = cv2.projectPoints(OBJP, rv, tv, K, d)[0].reshape(-1, 2)
    return float(np.sqrt(((uv - esq.reshape(-1, 2))**2).sum(1).mean()))


def extrinsecos(K, d, esq):
    """T_C0 a partir del tablero apoyado en la mesa (solvePnP). OpenCV numera las filas de esquinas desde arriba de la
    imagen, y la y de la mesa va hacia arriba en la imagen: con ese orden, el tablero sale «visto por detrás» (R33 > 0).
    Se prueban los dos órdenes y se elige el que deja la normal del tablero hacia la cámara (R33 < 0: la cámara mira
    hacia abajo)."""
    for orden in (esq, esq.reshape(NY, NX, 1, 2)[::-1].reshape(-1, 1, 2)):
        ok, rv, tv = cv2.solvePnP(OBJP, orden, K, d)
        R_ = cv2.Rodrigues(rv)[0]
        if R_[2, 2] < 0:
            break
    return Tmat(R_, tv.ravel()) @ Tinv(T_0TAB), orden is not esq


def a_la_mesa(uv, K, d, T_C0, z=0.0):
    """Del píxel a la mesa: el rayo K⁻¹·p̃ (sin la distorsión), girado al mundo y cortado con el plano de altura z."""
    n = cv2.undistortPointsIter(np.atleast_2d(uv).reshape(-1, 1, 2).astype(np.float64), K, d, None, None,
                                (cv2.TERM_CRITERIA_COUNT | cv2.TERM_CRITERIA_EPS, 100, 1e-12)).reshape(-1, 2)
    T0C = Tinv(T_C0)
    dir0 = (T0C[:3, :3] @ np.column_stack([n, np.ones(len(n))]).T).T
    o = T0C[:3, 3]
    return o + ((z - o[2])/dir0[:, 2])[:, None]*dir0


def inversa_2r(x, y, codo=+1):
    """La inversa del 2R de S10 (ley de cosenos y atan2), en grados."""
    c2 = (x*x + y*y - A1**2 - A2**2)/(2*A1*A2)
    q2 = codo*np.arccos(c2)
    q1 = np.arctan2(y, x) - np.arctan2(A2*np.sin(q2), A1 + A2*np.cos(q2))
    return q1*R2D, q2*R2D

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
assert EQUIPO in range(6), 'EQUIPO va de 1 a 5'
assert LETRA in list(LETRAS), "LETRA es una de 'A', 'B', 'C', 'D' o 'E', entre comillas"
t0 = time.time()
camara = Camara(EQUIPO, LETRA)                   # la unidad de vuestro equipo: hace las fotos, no dice su verdad
T_CAMARA = time.time() - t0                      # lo que tarda en prepararse [s]

In [ ]:
# @title ✔ Resultado
print(f'Equipo {camara.equipo}, letra {camara.letra} ({NOMBRE}): la unidad de cámara del equipo {camara.equipo}, lista en {coma(T_CAMARA, 1)} s.')
print('Hoja de datos: 640 x 480 px, píxel de 5 µm, óptica de 4 mm -> f_x = f_y = 800 px nominales, (c_x; c_y) = (320; 240).')
print('Plano del montaje: la cámara en (0,30; 0,20; 1,00) m sobre la mesa, mirando hacia abajo. La unidad real, hoy la medís.')

## 2. Fase 1: las quince vistas

*En clase, fase 1 · diapositiva 14*

Mejorad la tabla de partida, de frente y en el centro: inclinad de 20° a 40° en direcciones distintas, llevad el tablero a esquinas y bordes, y variad Z.
Mirad la detección y la cobertura; una vista sale con la hoja despegada.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# una fila por foto; el tablero tiene que caber entero en la imagen (640 x 480 px, origen arriba a la izquierda)
#   Z: distancia del centro del tablero a la cámara, de 0.35 a 0.6 m
#   inc_x, inc_y: inclinación alrededor del eje horizontal y del vertical de la imagen; (0, 0) es de frente
#   u, v: el píxel donde cae el centro del tablero · giro: el del tablero sobre sí mismo
# hasta dónde cabe, inclinado 30°: a 0.6 m, u de 160 a 480 y v de 120 a 360; a 0.35 m, u de 270 a 370 y v de 190 a 290
#          Z [m]  inc_x [°]  inc_y [°]   u [px]  v [px]  giro [°]
VISTAS = [(0.50,     0,         0,       320,    240,      0),
          (0.48,     1,         0,       310,    235,      5),
          (0.52,     0,         1,       330,    245,     -5),
          (0.50,    -1,         0,       325,    230,     10),
          (0.47,     0,        -1,       315,    250,    -10),
          (0.53,     1,         1,       320,    240,      3),
          (0.50,    -1,         1,       305,    240,     -3),
          (0.49,     1,        -1,       335,    240,      8),
          (0.51,     0,         0,       320,    225,     -8),
          (0.50,     0,         0,       320,    255,      0),
          (0.46,     1,         0,       300,    245,      6),
          (0.54,     0,         1,       340,    235,     -6),
          (0.50,    -1,        -1,       320,    240,     12),
          (0.48,     0,         0,       330,    250,    -12),
          (0.52,     1,         0,       310,    230,      0)]

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
assert len(VISTAS) == 15, 'La tabla tiene que tener quince vistas'
t0 = time.time()
fotos = camara.fotos_de(VISTAS)                  # las quince fotos (una, con la hoja despegada)
esquinas = [detecta(f) for f in fotos]           # las 54 esquinas de cada una, o None si no se detecta
T_FOTOS = time.time() - t0                       # lo que tarda [s]
CABE = [camara.cabe(fila) for fila in VISTAS]    # si el tablero cabe entero en la imagen

In [ ]:
# @title ✔ Resultado
print(f'Quince fotos y su detección en {coma(T_FOTOS, 1)} s.')
g = lambda a: f'{a:g}'.replace('.', ',').replace('-', '−')        # un número de la tabla, con coma decimal
for i, (fila, e) in enumerate(zip(VISTAS, esquinas)):
    aviso = '' if CABE[i] else '   <- se sale de la imagen'
    inc = '; '.join(g(a) + '°' for a in fila[1:3])
    print(f'  vista {i + 1:2d}: Z = {coma(fila[0])} m, inclinación ({inc}), centro en ({g(fila[3])}; {g(fila[4])}):'
          f' {"54 esquinas" if e is not None else "NO SE DETECTA"}{aviso}')
fig, axs = plt.subplots(3, 5, figsize=(13, 6.2))
for i, ax in enumerate(axs.ravel()):
    ax.imshow(fotos[i], cmap='gray', vmin=0, vmax=255)
    if esquinas[i] is not None:
        ax.plot(*esquinas[i].reshape(-1, 2).T, '.', color=IQS_VERDE, ms=2)
    ax.set_title(f'vista {i + 1}' + ('' if esquinas[i] is not None else ' · no se detecta'), fontsize=10, color='k' if esquinas[i] is not None else ROJO)
    ax.axis('off')
plt.tight_layout(); plt.show()
# la cobertura: dónde ha caído el tablero en cada vista
fig, ax = plt.subplots(figsize=(6.4, 4.8))
for e in esquinas:
    if e is not None:
        qe = e.reshape(NY, NX, 2)
        borde = np.vstack([qe[0, :], qe[:, -1], qe[-1, ::-1], qe[::-1, 0]])
        ax.fill(*borde.T, alpha=0.12, color=IQS_AZUL); ax.plot(*borde.T, color=IQS_AZUL, lw=0.8)
ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.set_aspect('equal'); ejes(ax)
ax.set_title('La cobertura de vuestras vistas: el tablero en cada una'); ax.set_xlabel('u [px]'); ax.set_ylabel('v [px]')
plt.show()

**Qué tiene que salir:** 15 de 15 detectadas y la cobertura por todo el campo, no apelotonada en el centro.

### Los tres fallos de siempre

Rasante, con un reflejo y cortado: `findChessboardCorners` necesita las 54 esquinas, y ninguna de las tres se detecta.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
trampas = camara.trampas()                       # las tres fotos que no hay que hacer
TRAMPAS_OK = [detecta(f) is not None for f in trampas]

In [ ]:
# @title ✔ Resultado
fig, axs = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, f, ok, nom in zip(axs, trampas, TRAMPAS_OK, ('rasante, a 82°', 'con un reflejo', 'cortado por el borde')):
    ax.imshow(f, cmap='gray', vmin=0, vmax=255); ax.axis('off')
    ax.set_title(f'{nom}: {"detectado" if ok else "no se detecta"}', fontsize=11, color=ROJO if not ok else 'k')
plt.tight_layout(); plt.show()

## 3. Fase 2: calibrar, depurar y validar

*En clase, fase 2 · diapositiva 15*

`calibra()` ajusta K, la distorsión y una pose por vista, minimizando el error de reproyección. RMS ≤ 0,5 px es necesario, pero no suficiente.

### Calibrar

Con todas las vistas: el RMS, K, la distorsión y el error de cada vista.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
usadas = [i for i, e in enumerate(esquinas) if e is not None]     # las vistas detectadas (desde 0)
assert len(usadas) >= 4, 'Hacen falta al menos cuatro vistas detectadas: volved a la tabla de la sección 2'
rms, K, dist, por_vista = calibra([esquinas[i] for i in usadas])

In [ ]:
# @title ✔ Resultado
def informe(rms, K, dist, por_vista, usadas, titulo):
    '''El RMS, K y la distorsión de una calibración y, en barras, el error de cada vista.'''
    print(f'{titulo}: {len(usadas)} vistas, RMS = {coma(rms, 3)} px')
    print(f'  f_x = {coma(K[0, 0], 1)} px, f_y = {coma(K[1, 1], 1)} px, (c_x; c_y) = ({coma(K[0, 2], 1)}; {coma(K[1, 2], 1)}) px')
    print(f'  k1 = {coma(dist[0], 4)}, k2 = {coma(dist[1], 4)}, p1 = {coma(dist[2], 5)}, p2 = {coma(dist[3], 5)}')
    fig, ax = plt.subplots(figsize=(9, 2.8))
    ax.bar([str(i + 1) for i in usadas], por_vista, color=[ROJO if e > 0.5 else IQS_AZUL for e in por_vista])
    ax.axhline(0.5, color=ROJO, ls='--', lw=1); ax.axhline(np.median(por_vista), color=GRIS, lw=1)
    ax.text(len(usadas) - 0.5, 0.5, ' 0,5 px', color=ROJO, va='bottom', ha='right', fontsize=9)
    ax.set_xlabel('vista'); ax.set_ylabel('RMS de la vista [px]'); ax.yaxis.set_major_formatter(COMA)
    ax.set_title(f'{titulo}: el error de cada vista (la gris, la mediana)'); plt.show()

informe(rms, K, dist, por_vista, usadas, 'Con todas')

**Qué tiene que salir:** con la tabla mejorada, RMS de 0,14 a 0,20 px y una vista de 0,37 a 0,66 px (la despegada) frente a una mediana de 0,08 a 0,12; con la de partida, ninguna pasa de 0,27 px: volved a la fase 1.

### Depurar

Quitad la vista que sobresale, con su motivo, y recalibrad. No quitéis vistas buenas solo para bajar el RMS.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# los números de las vistas que quitáis, por ejemplo [4]; con [], ninguna
QUITAR = []  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
assert isinstance(QUITAR, (list, tuple)), 'QUITAR va entre corchetes, por ejemplo [4]'
usadas = [i for i, e in enumerate(esquinas) if e is not None and i + 1 not in QUITAR]
assert len(usadas) >= 4, 'Quedan menos de cuatro vistas: quitad menos'
rms, K, dist, por_vista = calibra([esquinas[i] for i in usadas])

In [ ]:
# @title ✔ Resultado
informe(rms, K, dist, por_vista, usadas, 'Depurada' if QUITAR else 'Con todas (no habéis quitado ninguna)')

**Qué tiene que salir:** con la tabla mejorada y sin la despegada, RMS de 0,09 a 0,11 px.

### Validar fuera

Las dos vistas reservadas, que no entran en el ajuste; la hoja de datos, 800 px; y vuestros compañeros de equipo, que calibran la misma unidad con sus fotos: comparad en voz alta vuestras líneas de resumen.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
reservadas = [detecta(f) for f in camara.reservadas()]      # las dos vistas que no entran en el ajuste
err_res = [err_reservada(K, dist, e) for e in reservadas]  # su error, con vuestra K y vuestra distorsión fijas [px]
FX_PCT = 100*(K[0, 0] - 800)/800                           # f_x frente a los 800 px de la hoja de datos [%]
CENTRO_DESV = (K[0, 2] - 320, K[1, 2] - 240)               # el centro frente a (320; 240) [px]
CUMPLE_RMS, CUMPLE_RES = rms <= 0.5, max(err_res) <= 2*rms  # el criterio del curso y las reservadas

In [ ]:
# @title ✔ Resultado
print(f'Vistas reservadas: {coma(err_res[0], 3)} y {coma(err_res[1], 3)} px (el RMS del ajuste: {coma(rms, 3)} px).')
print(f'Hoja de datos: f_x sale un {coma(abs(FX_PCT), 1)} % {"por encima" if FX_PCT > 0 else "por debajo"} de los 800 px nominales;'
      f' el centro, a ({coma(CENTRO_DESV[0], 1)}; {coma(CENTRO_DESV[1], 1)}) px de (320; 240).')
print('RMS ≤ 0,5 px:', 'sí' if CUMPLE_RMS else 'NO', '· reservadas por debajo del doble del RMS:', 'sí' if CUMPLE_RES else 'NO')
print(f'\nPara decir en voz alta: E{EQUIPO}{LETRA} · RMS {coma(rms, 2)} px · f_x {coma(K[0, 0], 1)} px · centro ({coma(K[0, 2], 1)}; {coma(K[1, 2], 1)})'
      f' · reservadas {coma(err_res[0], 2)} y {coma(err_res[1], 2)} px')

**Qué tiene que salir:** reservadas de 0,07 a 0,13 px; las f_x de vuestros compañeros de equipo, a un píxel o poco más de la vuestra (a décimas, con la misma tabla). Con la tabla de partida, f_x de 1020 a 1350 px, o más de 2000: volved a la fase 1.

## 4. Fase 3: de la mesa al robot, y el fichero

*En clase, fase 3 · diapositiva 16 · los seis últimos minutos, para la lectura y para subirlo*

Los extrínsecos se miden en el montaje; con ellos, del píxel a la mesa y al 2R.

### Los extrínsecos, en el montaje

El tablero, apoyado en la marca (0,25; 0,10) m: `solvePnP` da T_C0. Si el orden de las esquinas sale al revés, el tablero se ve por detrás y la celda lo corrige.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
foto_mesa = camara.foto_mesa()                   # el tablero en la mesa, visto desde el montaje de verdad
esq_mesa = detecta(foto_mesa)
T_C0, invertido = extrinsecos(K, dist, esq_mesa)  # los extrínsecos, y si hubo que invertir el orden de las filas
T_0C = Tinv(T_C0)
o = T_0C[:3, 3]                                   # dónde está la cámara, en la mesa [m]
DESV_PLANO = 1e3*(o - np.array([0.30, 0.20, 1.00]))   # y lo que se aparta del plano del montaje [mm]
# para la figura: los ejes de la mesa, de 0.1 m, en el origen del tablero, proyectados en la imagen
uv_ej = cv2.projectPoints(np.array([[0, 0, 0], [0.1, 0, 0], [0, 0.1, 0], [0, 0, 0.1]]) + T_0TAB[:3, 3], cv2.Rodrigues(T_C0[:3, :3])[0], T_C0[:3, 3], K, dist)[0].reshape(-1, 2)

In [ ]:
# @title ✔ Resultado
print('El orden de las esquinas estaba al revés y se ha corregido.' if invertido else 'El orden de las esquinas era el bueno.')
print('Extrínsecos [R | t] = T_C0:')
for fila in T_C0[:3]:
    print('  ' + '   '.join(f'{coma(v, 4):>7s}' for v in fila))
print(f'La cámara está en ({coma(o[0], 4)}; {coma(o[1], 4)}; {coma(o[2], 4)}) m sobre la mesa: a'
      f' ({coma(DESV_PLANO[0], 1)}; {coma(DESV_PLANO[1], 1)}; {coma(DESV_PLANO[2], 1)}) mm de lo que dice el plano.')
print('Con una buena calibración, la altura se mide bien (a unos milímetros del plano, lo que se aparta el soporte); en x e y, la posición y el giro se compensan (con un tablero de 250 mm a 1 m, hasta 2 cm del plano): lo que cuenta son los mm en la mesa.')
fig, ax = plt.subplots(figsize=(6.4, 4.8))
ax.imshow(foto_mesa, cmap='gray', vmin=0, vmax=255); ax.axis('off')
for j, col in ((1, ROJO), (2, IQS_VERDE), (3, IQS_AZUL)):
    ax.annotate('', uv_ej[j], uv_ej[0], arrowprops=dict(arrowstyle='-|>', color=col, lw=2.2))
ax.set_title('El tablero en la mesa, con los ejes x (rojo), y (verde) y z (azul) de la mesa'); plt.show()

**Qué tiene que salir:** el orden, al revés y corregido; la altura, a menos de 8 mm de 1 m; en x e y, hasta 2 cm del plano. Con la tabla de partida, de 28 a 52 cm de más, o casi 2 m: es la alarma.

### Del píxel a la mesa, con su error en milímetros

Las 35 marcas del tapete, de posición conocida: el rayo de cada una, cortado con la mesa. Para comparar, con el plano del montaje y sin corregir la distorsión.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
uv_marcas = camara.mide(MARCAS)                  # el píxel de cada marca, con su ruido de medida

def error_mm(P):
    '''Lo que dista cada punto de la mesa de su marca [mm].'''
    return 1e3*np.linalg.norm(P[:, :2] - MARCAS[:, :2], axis=1)

e_med = error_mm(a_la_mesa(uv_marcas, K, dist, T_C0))                     # vuestra calibración y vuestros extrínsecos
e_plano = error_mm(a_la_mesa(uv_marcas, K, dist, Tinv(T_0C_NOMINAL)))     # con el plano del montaje
e_sindist = error_mm(a_la_mesa(uv_marcas, K, np.zeros(5), T_C0))          # sin corregir la distorsión

In [ ]:
# @title ✔ Resultado
for nom, e in (('vuestra calibración y los extrínsecos medidos', e_med), ('con el plano del montaje en lugar de los extrínsecos', e_plano),
               ('sin corregir la distorsión', e_sindist)):
    print(f'{nom:55s}: {coma(e.mean(), 2)} mm de media, {coma(e.max(), 2)} mm de máximo')
print(f'En la marca de la esquina, (0,6; 0,4) m: {coma(e_med[-1], 2)} mm.')

**Qué tiene que salir:** de 0,2 a 0,7 mm de media y de 0,4 a 2,3 de máximo; con el plano, de 10 a 14 mm; sin corregir la distorsión, de 3,5 a 5 mm.

### La pieza, y el 2R

Su cara de arriba está a 10 mm: cortar el rayo con la mesa es un error sistemático. La inversa de S10 da los ángulos.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
uv_pieza = camara.mide(P_PIEZA + [0, 0, H_PIEZA])[0]        # el píxel del centro de la cara de arriba
P_z0 = a_la_mesa(uv_pieza, K, dist, T_C0)[0]                # el rayo, cortado con la mesa
P_zh = a_la_mesa(uv_pieza, K, dist, T_C0, z=H_PIEZA)[0]     # y con la cara de la pieza
q = inversa_2r(P_zh[0], P_zh[1])                            # los ángulos del 2R, codo abajo [°]
DIF_PIEZA = 1e3*np.linalg.norm(P_z0[:2] - P_zh[:2])         # lo que se va la pieza si se corta con la mesa [mm]

In [ ]:
# @title ✔ Resultado
print(f'La pieza se ve en ({coma(uv_pieza[0], 1)}; {coma(uv_pieza[1], 1)}) px.')
print(f'  cortando con la mesa (z = 0):        ({coma(P_z0[0], 4)}; {coma(P_z0[1], 4)}) m')
print(f'  cortando con su cara (z = 0,010 m):  ({coma(P_zh[0], 4)}; {coma(P_zh[1], 4)}) m ; la diferencia, {coma(DIF_PIEZA, 2)} mm')
print(f'  la inversa de S10, codo abajo: q = ({coma(q[0], 2)}°; {coma(q[1], 2)}°)')

**Qué tiene que salir:** de 0,9 a 1,0 mm entre la mesa y la cara, y q ≈ (1,0°; 74,5°).

### El fichero de la calibración

`calibracion_E<n><letra>.json` (por ejemplo, `calibracion_E3C.json`): quién, qué cámara, cuándo y con qué vistas. El lunes 16, cada pareja del S17a carga el de uno de los dos; en la entrega 3, el equipo usa el de uno de sus miembros, el que mejor valide: guardadlo.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
assert (camara.equipo, camara.letra) == (EQUIPO, LETRA), 'Habéis cambiado EQUIPO o LETRA: volved a ejecutar desde la sección 1'
assert usadas == [i for i, e in enumerate(esquinas) if e is not None and i + 1 not in QUITAR], \
    'Habéis cambiado QUITAR: ejecutad el Cálculo de «Depurar» y lo que le sigue'
# 'pareja' guarda la letra y 'nombres', el nombre: son las claves de siempre, las que leen el S17a y la tabla de S20
entrega = {
    'curso': '82514 Mecatrónica y Robótica', 'sesion': 'S16', 'fecha': datetime.date.today().isoformat(),
    'equipo': EQUIPO, 'pareja': LETRA, 'nombres': NOMBRE, 'camara': f'unidad del equipo {EQUIPO}',
    'opencv': cv2.__version__, 'imagen_px': [W, H], 'tablero': {'esquinas': [NX, NY], 'lado_m': LADO},
    'K': np.round(K, 3).tolist(), 'distorsion_k1_k2_p1_p2_k3': np.round(dist, 6).tolist(), 'rms_px': round(float(rms), 4),
    'dist': np.round(dist, 6).tolist(), 'rms': round(float(rms), 4),          # las claves que lee el cuaderno S17a
    'vistas': [list(f) for f in VISTAS], 'vistas_usadas': [i + 1 for i in usadas], 'vistas_quitadas': list(QUITAR),
    'no_detectadas': [i + 1 for i, e in enumerate(esquinas) if e is None],
    'error_por_vista_px': {str(i + 1): round(float(e), 4) for i, e in zip(usadas, por_vista)},
    'reservadas_error_px': [round(e, 4) for e in err_res],
    'T_C0': np.round(T_C0, 5).tolist(), 'orden_esquinas_invertido': bool(invertido),
    'mesa_error_mm': {'medio': round(float(e_med.mean()), 3), 'maximo': round(float(e_med.max()), 3),
                      'marca_0,6_0,4': round(float(e_med[-1]), 3)},
    'pieza': {'pixel': np.round(uv_pieza, 2).tolist(), 'mesa_m': np.round(P_zh[:2], 5).tolist(), 'q_grados': [round(q[0], 3), round(q[1], 3)]},
}
nombre = f'calibracion_E{EQUIPO}{LETRA}.json'
ruta = nombre if EN_COLAB else os.path.join(tempfile.gettempdir(), nombre)

In [ ]:
# @title ✔ Resultado · el fichero de la entrega
with open(ruta, 'w', encoding='utf-8') as f:
    json.dump(entrega, f, ensure_ascii=False, indent=1)
print(f'ENTREGABLE S16 · E{EQUIPO}{LETRA} · {NOMBRE}')
print(f'  RMS {coma(rms, 3)} px con {len(usadas)} vistas (quitadas: {QUITAR or "ninguna"}) · reservadas {coma(err_res[0], 3)} y {coma(err_res[1], 3)} px')
print(f'  f_x {coma(K[0, 0], 1)} px · centro ({coma(K[0, 2], 1)}; {coma(K[1, 2], 1)}) · k1 {coma(dist[0], 3)} · mesa {coma(e_med.mean(), 2)} mm de media')
print('Escrito', ruta)
if EN_COLAB:
    from google.colab import files
    files.download(ruta)

### ✎ Vuestra lectura (tres líneas)

*Doble clic aquí; una línea por punto, con cifras. Después, Archivo → Descargar → Descargar .ipynb, y el .ipynb y el JSON, a la Tarea «Calibración de la cámara del equipo»: cada uno, los suyos.*

1. Cobertura: la tabla y la vista quitada, con su motivo.
2. RMS y reservadas: qué dice cada uno.
3. Verificación externa: hoja de datos, vuestros compañeros de equipo, mm en la mesa.

## 5. En casa: la cámara del curso, por dentro

*En casa*

La unidad 0, la de la hoja de datos exacta (f = 800 px, k₁ = −0,25, k₂ = 0,06): las cuentas de las diapositivas 5 a 10.

### 5.1 Del punto de la mesa al píxel (diapositivas 5 a 8)

[R | t] = T_C0 = T_0C⁻¹ lleva P a la cámara; K, al píxel, dividiendo por la tercera componente.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
K_CURSO = np.array([[800., 0, 320], [0, 800., 240], [0, 0, 1]])   # la cámara del curso: la hoja de datos exacta
DIST_CURSO = np.array([-0.25, 0.06, 0, 0, 0])
T_C0_plano = Tinv(T_0C_NOMINAL)                  # los extrínsecos del plano del montaje: [R | t]
PROY = []                                        # (nombre, P en la cámara, K·P, píxel) de la pieza y de la base del 2R
for nombre_p, P0 in (('P, la pieza', P_PIEZA), ('la base del 2R', np.zeros(3))):
    Pc = T_C0_plano[:3, :3] @ P0 + T_C0_plano[:3, 3]
    p = K_CURSO @ Pc
    PROY.append((nombre_p, Pc, p, p[:2]/p[2]))
CAMPO = (2*np.degrees(np.arctan(320/800)), 2*np.degrees(np.arctan(240/800)))   # el campo de visión [°]
MM_PX = 1/800*1e3                                # lo que mide un píxel sobre la mesa a 1 m [mm]

In [ ]:
# @title ✔ Resultado
print('[R | t] = T_C0 =')
for fila in T_C0_plano[:3]:
    print('  ' + '   '.join(f'{coma(v, 3):>6s}' for v in fila))
for nombre_p, Pc, p, px in PROY:
    print(f'{nombre_p}: en la cámara ({coma(Pc[0])}; {coma(Pc[1])}; {coma(Pc[2])}) m -> K·P = ({coma(p[0], 1)}; {coma(p[1], 1)}; {coma(p[2])})'
          f' -> píxel ({coma(px[0], 1)}; {coma(px[1], 1)})')
print(f'Campo de visión: {coma(CAMPO[0], 1)}° x {coma(CAMPO[1], 1)}°; a 1 m, 1 px = {coma(MM_PX, 3)} mm sobre la mesa.')

**Qué tiene que salir:** P en (400; 240), la base del 2R en (80; 400), campo de 43,6° × 33,4° y 1,25 mm por píxel.

### 5.2 La profundidad que se pierde (diapositiva 6)

λ·P cae en el mismo píxel. Con dos cámaras a una base b, z = f·b/d (S6).

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# los factores por los que se multiplica P, en la cámara
LAMBDAS = [1, 2, 5]  # @param {type:"raw"}
# dos discos: (diámetro [m], distancia a la cámara [m])
DISCOS = [(0.040, 1.0), (0.080, 2.0)]  # @param {type:"raw"}
# dos cámaras del curso: la base entre ellas [m] y la distancia [m]
B_ESTEREO = 0.10  # @param {type:"number"}
Z_ESTEREO = 1.0  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
Pc = T_C0_plano[:3, :3] @ P_PIEZA + T_C0_plano[:3, 3]           # P, en la cámara
ESCALAS = []                                                    # (λ, K·(λ·P), píxel)
for lam in LAMBDAS:
    p = K_CURSO @ (lam*Pc)
    ESCALAS.append((lam, p, p[:2]/p[2]))
TAMANOS = [(D, Z, 800*D/Z) for D, Z in DISCOS]                  # el diámetro en la imagen, f·D/Z [px]
D_ESTEREO = 800*B_ESTEREO/Z_ESTEREO                             # el desplazamiento entre las dos imágenes, f·b/z [px]

In [ ]:
# @title ✔ Resultado
for lam, p, px in ESCALAS:
    print(f'λ = {lam}: K·(λ·P) = ({coma(p[0], 1)}; {coma(p[1], 1)}; {coma(p[2], 1)}) -> píxel ({coma(px[0], 1)}; {coma(px[1], 1)})')
for D, Z, dpx in TAMANOS:
    print(f'Un disco de {coma(1e3*D, 0)} mm a {coma(Z, 1)} m: {coma(dpx, 1)} px de diámetro en la imagen')
b_cm, z_m = (f'{v:g}'.replace('.', ',') for v in (100*B_ESTEREO, Z_ESTEREO))   # la base y la distancia, con coma
print(f'Dos cámaras del curso a {b_cm} cm, a {z_m} m: d = f·b/z = {coma(D_ESTEREO, 0)} px')

**Qué tiene que salir:** (400; 240) con λ = 1, 2 y 5; los dos discos, 32 px; d = 80 px.

### 5.3 La distorsión (diapositiva 9)

x_d = x_n·(1 + k₁·r² + k₂·r⁴), con p₁ = p₂ = 0. `cv2.undistort` endereza la imagen entera.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# distancias al centro de la imagen que se prueban [px]
RADIOS_PX = [80, 200, 320, 400]  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
DESPL = []                                       # (px del centro, r, factor, desplazamiento hacia dentro [px])
for upx in RADIOS_PX:
    r = upx/800
    fac = 1 + DIST_CURSO[0]*r**2 + DIST_CURSO[1]*r**4
    DESPL.append((upx, r, fac, upx*(1 - fac)))
# una rejilla de rectas, vista con la lente del curso, y después corregida
ideal = np.full((H, W), 235, np.uint8)
for x in range(0, W + 1, 40):
    cv2.line(ideal, (x, 0), (x, H), 40, 2)
for y in range(0, H + 1, 40):
    cv2.line(ideal, (0, y), (W, y), 40, 2)
mx, my = cv2.initInverseRectificationMap(K_CURSO, DIST_CURSO, None, K_CURSO, (W, H), cv2.CV_32FC1)
curva = cv2.remap(ideal, mx, my, cv2.INTER_LINEAR, borderValue=180)
recta = cv2.undistort(curva, K_CURSO, DIST_CURSO)

In [ ]:
# @title ✔ Resultado
for upx, r, fac, d in DESPL:
    print(f'a {upx} px del centro (r = {coma(r, 3)}): factor {coma(fac, 4)}, se desplaza {coma(d, 2)} px')
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].imshow(curva, cmap='gray'); axs[0].set_title('Con la lente: las rectas se curvan hacia los bordes')
axs[1].imshow(recta, cmap='gray'); axs[1].set_title('Con cv2.undistort: rectas otra vez')
for ax in axs:
    ax.axis('off')
plt.tight_layout(); plt.show()

**Qué tiene que salir:** 0,20, 3,08, 12,31 y 23,50 px hacia dentro a 80, 200, 320 y 400 px.

### 5.4 Ampliación voluntaria: una cámara real

Las trece fotos de muestra de OpenCV (licencia Apache 2.0), si hay red; o, con `MIAS = True`, las vuestras: un tablero de 9 × 6 esquinas en cartón y unas quince fotos con el zum fijo.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# True: subir vuestras fotos (en Colab); False: las trece de muestra de OpenCV
MIAS = False  # @param {type:"boolean"}
# el lado del cuadro de vuestro tablero, medido con una regla [m]
LADO_MIO = 0.025  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
grises, AVISO_RED = [], None                     # las fotos, en gris; y el motivo, si no hay red
if MIAS and EN_COLAB:
    from google.colab import files
    subidas = files.upload()
    grises = [cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_GRAYSCALE) for b in subidas.values()]
else:
    base = 'https://raw.githubusercontent.com/opencv/opencv/4.14.0/samples/data/left{:02d}.jpg'
    try:
        for i in (1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14):
            datos = urllib.request.urlopen(base.format(i), timeout=10).read()
            grises.append(cv2.imdecode(np.frombuffer(datos, np.uint8), cv2.IMREAD_GRAYSCALE))
    except Exception as ex:
        AVISO_RED, grises = type(ex).__name__, []
ok_reales = []                                   # las esquinas de las fotos con el tablero entero
for g in grises:
    hallado, e = cv2.findChessboardCorners(g, (NX, NY))
    if hallado:
        ok_reales.append(cv2.cornerSubPix(g, e, (11, 11), (-1, -1), CRIT))
REAL = None                                      # (RMS, K, distorsión, error por vista) de la cámara real
if len(ok_reales) >= 3 and grises[0].shape == (H, W):
    obj = [(OBJP*(LADO_MIO/LADO if MIAS else 1)).astype(np.float32)]*len(ok_reales)
    sal = cv2.calibrateCameraExtended(obj, ok_reales, (W, H), None, None, flags=cv2.CALIB_FIX_K3)
    REAL = (sal[0], sal[1], sal[2].ravel(), sal[-1].ravel())

In [ ]:
# @title ✔ Resultado
if AVISO_RED:
    print('Sin red para las imágenes de muestra:', AVISO_RED)
if REAL is not None:
    rms_r, K_r, d_r, pv_r = REAL
    print(f'{len(ok_reales)} de {len(grises)} fotos con el tablero entero: RMS {coma(rms_r, 3)} px, f_x {coma(K_r[0, 0], 1)} px,'
          f' centro ({coma(K_r[0, 2], 1)}; {coma(K_r[1, 2], 1)}), k1 {coma(d_r[0], 3)}')
    print('Error por vista:', '; '.join(coma(e, 2) for e in pv_r), 'px')
elif grises:
    print('Hacen falta al menos tres fotos de 640 x 480 con el tablero entero (con otra resolución, cambiad W y H).')

**Qué tiene que salir:** 13 de 13 fotos, RMS 0,41 px y una vista de 1,22 px: una cámara real.

## 6. En casa: tres ejercicios

*En casa; las soluciones, al final*

### Ejercicio 1 (en casa): otra óptica

Con el mismo sensor y una óptica de 6 mm, ¿cuánto valen f_x, el campo y lo que mide un píxel en la mesa a 1 m? Desde el mismo sitio, ¿se ven la pieza y la base del 2R?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# la óptica nueva [m]
E1_OPTICA = 0.006  # @param {type:"number"}

In [ ]:
# @title ✎ Completad · el hueco está marcado con None
# f_x con la óptica nueva [px]: la focal entre el tamaño del píxel, RHO
E1_FX = None  # ← completad

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
E1 = None                                        # (campo [°], mm por píxel a 1 m, los dos puntos)
if E1_FX is not None:
    E1_K = np.array([[E1_FX, 0, 320], [0, E1_FX, 240], [0, 0, 1]])
    E1_PUNTOS = []                               # (nombre, píxel, si cae dentro de la imagen)
    for e1_nom, e1_P0 in (('P, la pieza', P_PIEZA), ('la base del 2R', np.zeros(3))):
        e1_p = E1_K @ (T_C0_plano[:3, :3] @ e1_P0 + T_C0_plano[:3, 3])
        e1_uv = e1_p[:2]/e1_p[2]
        E1_PUNTOS.append((e1_nom, e1_uv, bool(0 <= e1_uv[0] <= W and 0 <= e1_uv[1] <= H)))
    E1 = ((2*np.degrees(np.arctan(320/E1_FX)), 2*np.degrees(np.arctan(240/E1_FX))), 1e3/E1_FX, E1_PUNTOS)

In [ ]:
# @title ✔ Resultado
if E1 is None:
    print('Pendiente: completad E1_FX en la celda ✎')
else:
    (campo_x, campo_y), mm_px, puntos = E1
    print(f'f_x = {coma(E1_FX, 0)} px · campo de {coma(campo_x, 1)}° x {coma(campo_y, 1)}° · a 1 m, 1 px = {coma(mm_px, 2)} mm')
    for nom, uv, dentro in puntos:
        print(f'{nom}: píxel ({coma(uv[0], 1)}; {coma(uv[1], 1)}), {"dentro" if dentro else "fuera"} de la imagen')

### Ejercicio 2 (en casa): lo que estropea una calibración

Con la cámara del curso (`Camara(0, 'A')`, 800 px), calibrad con (a) tres vistas inclinadas; (b) diez casi de frente (menos de 3°) por toda la imagen; (c) la tabla de partida. ¿Importa más el número de vistas o su variedad?

In [ ]:
# @title ✎ Completad · el hueco está marcado con None
# tres tablas como VISTAS: una lista de filas (Z, inc_x, inc_y, u, v, giro)
# (a) tres vistas bien inclinadas, en direcciones distintas
E2_A = None  # ← completad
# (b) diez vistas casi de frente (menos de 3°), repartidas por toda la imagen
E2_B = None  # ← completad
# (c) la tabla de partida de la sección 2
E2_C = None  # ← completad

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
E2 = {}                                          # caso -> lo que sale de calibrar con su tabla
for caso, tabla in (('a', E2_A), ('b', E2_B), ('c', E2_C)):
    if tabla is None:                            # ese hueco sigue vacío
        continue
    e2_cam = Camara(0, 'A')                      # la cámara del curso: su verdad es la hoja de datos
    e2_esq = [e for e in (detecta(e2_cam.foto(*e2_cam.pose(f))) for f in tabla) if e is not None]
    E2[caso] = {'cam': e2_cam, 'detectadas': len(e2_esq), 'de': len(tabla), 'rms': None}
    if len(e2_esq) >= 3:                         # con menos de tres vistas detectadas no se calibra
        e2_rms, e2_K, e2_d, _ = calibra(e2_esq)
        E2[caso].update(rms=e2_rms, K=e2_K, d=e2_d, fx_pct=100*(e2_K[0, 0] - 800)/800,
                        reservadas=[err_reservada(e2_K, e2_d, detecta(f)) for f in e2_cam.reservadas()])

In [ ]:
# @title ✔ Resultado
if not E2:
    print('Pendiente: completad E2_A, E2_B o E2_C en la celda ✎')
for caso, r2 in E2.items():
    if r2['rms'] is None:
        print(f"({caso}) {r2['detectadas']} de {r2['de']} vistas detectadas: hacen falta al menos tres")
    else:
        print(f"({caso}) {r2['detectadas']} de {r2['de']} vistas · RMS {coma(r2['rms'], 3)} px · f_x {coma(r2['K'][0, 0], 1)} px"
              f" ({coma(r2['fx_pct'], 1)} %) · reservadas {coma(r2['reservadas'][0], 2)} y {coma(r2['reservadas'][1], 2)} px")

### Ejercicio 3 (en casa): los extrínsecos esconden la focal

Con la calibración (b): ¿es peor el error en las marcas? ¿A qué altura pone `solvePnP` la cámara, que está a 1,000 m? Explicadlo con u = f_x·X/Z.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
E3 = {}                                          # caso -> (altura de la cámara [m], error medio en las marcas [mm])
for caso, r2 in E2.items():
    if r2['rms'] is None:
        continue
    e3_T, _ = extrinsecos(r2['K'], r2['d'], detecta(r2['cam'].foto_mesa()))
    e3_err = 1e3*np.linalg.norm(a_la_mesa(r2['cam'].mide(MARCAS), r2['K'], r2['d'], e3_T)[:, :2] - MARCAS[:, :2], axis=1)
    E3[caso] = (Tinv(e3_T)[2, 3], e3_err.mean())

In [ ]:
# @title ✔ Resultado
if 'b' not in E3:
    print('Pendiente: completad E2_B, la tabla (b) del ejercicio 2')
for caso, (altura, err) in E3.items():
    print(f'({caso}) la cámara, a {coma(altura, 3)} m (la verdad: 1,000 m) · marcas: {coma(err, 2)} mm de media')

---

## Soluciones

**Fase 1 (sección 2): una tabla posible.** Quince vistas inclinadas de 28° a 41° en direcciones distintas, con el centro por toda la imagen y Z de 0,4 a 0,6 m; con todas las unidades se detectan las quince:

```
VISTAS = [(0.45, 30, 0, 230, 170, 10), (0.45, -30, 0, 410, 310, -10), (0.45, 0, 30, 220, 310, 5), (0.45, 0, -30, 420, 170, -5),
          (0.50, 25, 25, 320, 240, 20), (0.50, -25, -25, 320, 240, -20), (0.55, 35, -15, 190, 240, 0), (0.55, -35, 15, 450, 240, 0),
          (0.40, 20, -20, 320, 200, 15), (0.40, -20, 20, 320, 280, -15), (0.60, 15, 35, 220, 330, 10), (0.60, -15, -35, 440, 130, -10),
          (0.50, 40, 10, 260, 200, 0), (0.50, -10, 40, 380, 280, 25), (0.45, 10, -40, 330, 250, -25)]
```

No es la única: lo que importa es que las vistas estén inclinadas y cubran el campo.

**Fase 2 (sección 3).** Con esa tabla, RMS de 0,14 a 0,20 px con todas y una vista de 0,37 a 0,66 px (la despegada; su número cambia con la letra) frente a una mediana de 0,08 a 0,12. Sin ella, RMS de 0,09 a 0,11 px, reservadas de 0,07 a 0,13 px y los de un mismo equipo, a unas décimas de píxel en f_x (como mucho, 0,6 px). Con la tabla de partida, RMS de 0,12 a 0,15 px, pero f_x de 1020 a 1350 px (un 28 % o más) y reservadas de 1,7 a 3 px; con alguna letra, f_x pasa de 2000 px y las reservadas, de 5 px: el RMS no lo ve; la hoja de datos y las reservadas, sí.

**Fase 3 (sección 4).** El orden de las esquinas sale al revés en todas las unidades. La altura, a menos de 8 mm de 1 m (a 1,4 mm como mucho de la verdadera); en x e y, hasta 2 cm del plano, porque con un tablero de 250 mm a 1 m la posición lateral y el giro se compensan. Las marcas, de 0,2 a 0,7 mm de media y de 0,4 a 2,3 de máximo; con el plano del montaje, de 10 a 14 mm: por eso se miden; sin corregir la distorsión, de 3,5 a 5 mm de media y de 11 a 15 de máximo. La pieza, de 0,9 a 1,0 mm entre la mesa y su cara, y q ≈ (1,0°; 74,5°). Con la tabla de partida, las marcas salen igual de bien, porque los extrínsecos compensan la focal, pero la cámara sale de 28 a 52 cm más alta (con alguna letra, casi 2 m): es la alarma (con la cámara del curso, el ejercicio 2(c)).

**Ejercicio 1.** `E1_FX = E1_OPTICA/RHO`: 1200 px; campo de 29,9° × 22,6° y 0,83 mm por píxel a 1 m. P cae en (440; 240), dentro; la base del 2R, en (−40; 480), fuera. Más focal, más detalle y menos campo: o no se ve la base, o se sube la cámara a 1,5 m.

**Ejercicio 2.** (a) Por ejemplo, tres filas de la tabla de la fase 1 con inclinaciones distintas: RMS de 0,07 a 0,12 px, f_x a menos de 2 px de 800 y reservadas de 0,07 a 0,14 px; tres vistas variadas bastan. (b) Diez con inc_x e inc_y de −2 a 2: RMS de unos 0,1 px, pero f_x sin fijar, de un 0,3 % a un 31 % de error en veinte tablas al azar (831 px, +4 %, en la diapositiva 12), y reservadas de hasta unas cuarenta veces el RMS. (c) RMS de 0,15 a 0,22 px, pero f_x de unos 900 a casi 5000 px y reservadas de 2,5 a 7 px. Importa la variedad: sin inclinación, acercar el tablero o alargar la focal dan la misma imagen.

**Ejercicio 3.** Las marcas salen como con una buena (en torno a medio milímetro), pero `solvePnP` pone la cámara más alta o más baja en la proporción en que se equivoca f_x: con 831 px (+4 %), a 1,037 m. Las esquinas en la mesa fijan f_x/Z (u − c_x = f_x·X/Z): si f_x sale un 4 % alta, Z también; en el plano de la mesa todo cuadra, fuera de él no. Es la pose que se dará a la cámara en ROS 2.

**Ampliación (sección 5.4).** RMS 0,41 px, f_x 536,5 px, centro (342,4; 235,5) y k₁ = −0,28. Una vista pasa de 1 px (1,22): se quita solo si se ve el motivo. Cumple el criterio de 0,5 px.

---

## Para llevarse de esta sesión

- **Un píxel es un rayo:** de vuelta a la mesa, hace falta un plano.
- **Calibrar es ajustar por mínimos cuadrados;** el RMS mide el ajuste, no la exactitud.
- **Las vistas lo deciden:** inclinadas y por todo el campo; de frente, la focal sale mal.
- **La exactitud se comprueba fuera:** reservadas, hoja de datos, los compañeros de equipo y mm en la mesa.
- **Los intrínsecos valen para cualquier montaje; los extrínsecos, solo para el suyo.**

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*